In [1]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

C:\Users\hope\AppData\Local\Temp\ipykernel_30384\3336491994.py:7: DeprecationWarning: 'asyncio.get_event_loop_policy' is deprecated and slated for removal in Python 3.16
  if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
C:\Users\hope\AppData\Local\Temp\ipykernel_30384\3336491994.py:7: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
C:\Users\hope\AppData\Local\Temp\ipykernel_30384\3336491994.py:8: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
C:\Users\hope\AppData\Local\Temp\ipykernel_30384\3336491994.py:8: DeprecationWarning: 'asyncio.set_event_loop_policy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.

In [2]:
# 1.加载环境变量
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient

load_dotenv()

# 2.初始化模型
model = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

# 3.定义工具，用MCP获取工具

# 3.1.定义mcp client
client = MultiServerMCPClient(
    {
        "time-mcp": {
            "transport": "stdio",
            "args": [
                "-y",
                "time-mcp"
            ],
            "command": "npx"
        },
        "12306-mcp": {
          "transport": "streamable_http",
          "url": "https://mcp.api-inference.modelscope.net/50719920b4f24d/mcp"
        }
    }
)
# 3.2.用client拉取tool
tools = await client.get_tools()

# 4.创建Agent，绑定模型和工具
agent = create_agent(
    model=model,
    tools=tools
)

# 5.invoke调用Agent， 传入的Dict，里面有messages
response = await agent.ainvoke(
    {"messages": [HumanMessage("帮我查一下下周四从北京到杭州的高铁")]}
)
print(response)

{'messages': [HumanMessage(content='帮我查一下下周四从北京到杭州的高铁', additional_kwargs={}, response_metadata={}, id='a1495ab5-e094-439f-ba63-b16ecf6340b5'), AIMessage(content="I'll help you check high-speed rail tickets from Beijing to Hangzhou for next Thursday. Let me first get the current date to determine the date for next Thursday.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 3191, 'total_tokens': 3251, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 3072}, 'prompt_cache_hit_tokens': 3072, 'prompt_cache_miss_tokens': 119}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_a18b46594c_prod0820_fp8_kvcache_20260402', 'id': '349de235-f9e3-4934-ba80-688682706930', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ff370-8a17-7003-b55d-1942bb2221ca-0', tool_calls=[{'name': 'get-current-date', 'arg

In [3]:
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

帮我查一下下周四从北京到杭州的高铁
================================== Ai Message ==================================

I'll help you check high-speed rail tickets from Beijing to Hangzhou for next Thursday. Let me first get the current date to determine the date for next Thursday.
Tool Calls:
  get-current-date (call_00_hLpmuPbiKijeRZM8wXjD2966)
 Call ID: call_00_hLpmuPbiKijeRZM8wXjD2966
  Args:
================================= Tool Message =================================
Name: get-current-date

[{'type': 'text', 'text': '2026-08-12', 'id': 'lc_f21df87a-96b8-487e-b349-602a9004b567'}]
================================== Ai Message ==================================

Today is 2026-08-12 (Wednesday). Next Thursday would be 2026-08-20. Let me get the station codes for Beijing and Hangzhou, and check the tickets.
Tool Calls:
  get-station-code-of-citys (call_00_Z510EFo8TLrMneFAqKuK6706)
 Call ID: call_00_Z510EFo8TLrMneFAqKuK67